# Fact Layer v_n — Full-Corpus Stats (run on the MERGED folder)

Computes, from the merged per-case JSONs only (no API calls): integrity, field completeness (with 95% CIs),
outcome, statute, citation, concept, argument and party metrics, plus heuristic *review candidates*.

**Outputs** (written to `REPORT_DIR`, upload these two to Claude for the full report):
- `chunk_stats.csv` — one row per case
- `summary_metrics.json` — every aggregate table

**Read this first:** field paths below are best guesses from the handoff docs. Cell 3 prints which paths actually
resolved. If a logical field shows `0 resolved`, fix `FIELD_PATHS` in Cell 1 and re-run from Cell 4.

## Cell 1 — Setup + config

In [ ]:
import os, json, re, math
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print("Not running in Colab - using local paths")

MERGED_DIR = os.environ.get("MERGED_DIR", "/content/drive/MyDrive/MERGED_fact_layer_final_v2")
REPORT_DIR = os.environ.get("REPORT_DIR", "/content/drive/MyDrive/monitoring_report")
os.makedirs(REPORT_DIR, exist_ok=True)

SKIP_NAMES = {"done_keys.json", "errors.json", "checkpoint.json", "token_usage.json", "merge_manifest.json"}
N_WORKERS = 16

# Approximate chunk bands (real chunks split by count, so boundary years straddle two chunks)
YEAR_BANDS = [("2000-06", 2000, 2006), ("2007-13", 2007, 2013), ("2014-19", 2014, 2019), ("2020-25", 2020, 2025)]

# Logical field -> candidate dotted paths, first non-null wins. EDIT if Cell 3 shows a field unresolved.
FIELD_PATHS = {
    "title":            ["title"],
    "filename":         ["filename", "metadata.filename"],
    "method":           ["extraction_method", "metadata.extraction_method"],
    "issue":            ["issue"],
    "facts":            ["facts"],
    "pet_args":         ["petitioner_arguments", "arguments.petitioner"],
    "res_args":         ["respondent_arguments", "arguments.respondent"],
    "parties":          ["parties", "metadata.parties"],
    "statutes":         ["statutes", "reasoning.statutes"],
    "citations":        ["citations", "reasoning.citations"],
    "concepts":         ["legal_concepts", "reasoning.legal_concepts"],
    "principles":       ["reasoning.principles", "reasoning_principles", "principles"],
    "reasoning_summary":["reasoning.summary", "reasoning_summary"],
    "outcome":          ["decision.outcome", "outcome", "decision_outcome"],
    "outcome_details":  ["decision.details", "decision_details", "decision.decision_details"],
    "outcome_reasoning":["decision.outcome_reasoning", "outcome_reasoning"],
    "summary":          ["summary", "case_summary"],
    "reasoning_len":    ["_debug_reasoning_text_len", "metadata._debug_reasoning_text_len"],
}
print("MERGED_DIR:", MERGED_DIR)

## Cell 2 — Load every merged JSON

In [ ]:
files = sorted(f for f in os.listdir(MERGED_DIR) if f.endswith(".json") and f not in SKIP_NAMES)
print("JSON files found:", len(files))

def _load(f):
    try:
        with open(os.path.join(MERGED_DIR, f), encoding="utf-8") as fh:
            return f, json.load(fh), None
    except Exception as e:
        return f, None, repr(e)

records, load_errors = {}, []
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    for f, rec, err in tqdm(ex.map(_load, files), total=len(files)):
        if err or not isinstance(rec, dict): load_errors.append((f, err or "not a dict"))
        else: records[f] = rec
print(f"Loaded {len(records)} records, {len(load_errors)} failed to load")

## Cell 3 — Schema check: which field paths actually resolved?

In [ ]:
def dig(rec, path):
    cur = rec
    for p in path.split("."):
        if isinstance(cur, dict) and p in cur: cur = cur[p]
        else: return None
    return cur

def get(rec, name):
    for p in FIELD_PATHS[name]:
        v = dig(rec, p)
        if v is not None: return v, p
    return None, None

res_count = {k: Counter() for k in FIELD_PATHS}
for rec in records.values():
    for k in FIELD_PATHS:
        _, p = get(rec, k)
        res_count[k][p] += 1

print(f"{'field':20s} resolved  paths used")
for k, c in res_count.items():
    n_ok = sum(v for p, v in c.items() if p is not None)
    flag = "   <-- FIX FIELD_PATHS" if n_ok == 0 else ""
    print(f"{k:20s} {n_ok:7d}  { {p: v for p, v in c.items() if p} }{flag}")

print("\nTop-level keys seen (count):")
print(Counter(k for r in records.values() for k in r).most_common(40))

## Cell 4 — Build the per-case table

In [ ]:
NAMED_ACT_RE = re.compile(r"\b(Act|Code|Constitution|Rules?|Regulations?|Order|Ordinance|Schedule|IPC|CrPC|CPC|Penal|Procedure|Evidence)\b", re.I)
BARE_SECTION_RE = re.compile(r"(?i)^\s*(section|sec\.?)[\s_]*\d+[a-z]{0,2}(\(\w+\))*\s*$")
YEAR_TOKEN_RE = re.compile(r"^(18|19|20)\d{2}$")
NUM_RE = re.compile(r"\b(\d{1,4}[A-Za-z]{0,2})\b")
NAMED_CITE_RE = re.compile(r"\b(v\.?|vs\.?|versus)\s", re.I)
REPORTER_RE = re.compile(r"\b(SCC|AIR|SCR|SCALE|JT)\b", re.I)
NULLISH = {"", "null", "none", "n/a", "na", "unknown", "not available", "not specified"}

def item_text(x):
    if isinstance(x, str): return x
    if isinstance(x, dict):
        for k in ("name", "statute", "citation", "text", "title", "concept", "argument"):
            if isinstance(x.get(k), str): return x[k]
        return json.dumps(x, ensure_ascii=False)
    return str(x)

def as_list(v):
    if v is None: return []
    if isinstance(v, list): return [x for x in v if not (isinstance(x, str) and x.strip().lower() in NULLISH)]
    if isinstance(v, str): return [] if v.strip().lower() in NULLISH else [v]
    if isinstance(v, dict): return list(v.values())
    return [v]

def text_len(v):
    if v is None: return 0
    if isinstance(v, str): return 0 if v.strip().lower() in NULLISH else len(v)
    if isinstance(v, list): return sum(text_len(x) for x in v)
    if isinstance(v, dict): return sum(text_len(x) for x in v.values())
    return len(str(v))

def nums(s):
    return {t.lower() for t in NUM_RE.findall(s) if not YEAR_TOKEN_RE.match(t)}

def party_name(p):
    if p is None: return ""
    if isinstance(p, str): return p
    if isinstance(p, dict): return item_text(p.get("name", p))
    if isinstance(p, list): return "; ".join(party_name(x) for x in p)
    return str(p)

def band_of(year):
    for name, lo, hi in YEAR_BANDS:
        if year is not None and lo <= year <= hi: return name
    return "other"

DISMISS_RE = re.compile(r"\b(dismiss\w*|rejected|no merit|devoid of merit|without merit)\b", re.I)
ALLOW_RE   = re.compile(r"\b(allowed|set aside|quashed|acquitted)\b", re.I)

rows = []
for fname, rec in records.items():
    g = lambda k: get(rec, k)[0]
    case_id = rec.get("case_id", "")
    m = re.match(r"(\d{4})_", str(case_id)); year = int(m.group(1)) if m else None
    method = str(g("method") or "")
    outcome = g("outcome"); outcome = None if (outcome is None or str(outcome).strip().lower() in NULLISH) else str(outcome).strip()
    details = g("outcome_details"); details_txt = details if isinstance(details, str) else json.dumps(details or "")

    stat = [item_text(x) for x in as_list(g("statutes"))]
    bare = [s for s in stat if BARE_SECTION_RE.match(s)]
    named = [s for s in stat if s not in bare and NAMED_ACT_RE.search(s)]
    named_nums = set().union(*[nums(s) for s in named]) if named else set()
    bare_dup = [s for s in bare if nums(s) and nums(s) <= named_nums]

    cites = [item_text(x) for x in as_list(g("citations"))]
    cite_named = [c for c in cites if NAMED_CITE_RE.search(c)]
    cite_bare_rep = [c for c in cites if not NAMED_CITE_RE.search(c) and REPORTER_RE.search(c)]

    concepts = [item_text(x) for x in as_list(g("concepts"))]
    pet = as_list(g("pet_args")); res = as_list(g("res_args"))
    principles = as_list(g("principles"))

    parties = g("parties"); pp = rn = ""
    if isinstance(parties, dict):
        pp, rn = party_name(parties.get("petitioner")), party_name(parties.get("respondent"))
    title = str(g("title") or "")
    swap_cand = None
    sides = re.split(r"\s+(?:vs\.?|v\.?|versus)\s+", title, maxsplit=1, flags=re.I)
    if len(sides) == 2 and pp and rn:
        tok = lambda s: {w for w in re.findall(r"[a-z]{4,}", s.lower())}
        L, R, P, Rn = tok(sides[0]), tok(sides[1]), tok(pp), tok(rn)
        if P and Rn:
            swap_cand = bool((P & R) and not (P & L) and (Rn & L) and not (Rn & R))

    inv = None
    if outcome == "Accept" and DISMISS_RE.search(details_txt) and not ALLOW_RE.search(details_txt): inv = "Accept_but_text_says_dismissed"
    if outcome == "Reject" and ALLOW_RE.search(details_txt) and not DISMISS_RE.search(details_txt): inv = "Reject_but_text_says_allowed"

    rows.append(dict(
        case_id=case_id, filename=fname, year=year, band=band_of(year), method=method,
        is_fallback=("regex_fallback" in method), call1_ok=("case:groq" in method), call2_ok=("analysis:groq" in method),
        title_present=bool(title.strip()),
        issue_present=text_len(g("issue")) > 0, facts_present=text_len(g("facts")) > 0, facts_chars=text_len(g("facts")),
        summary_present=text_len(g("summary")) > 0, reasoning_summary_present=text_len(g("reasoning_summary")) > 0,
        principles_n=len(principles), principles_present=len(principles) > 0,
        outcome=outcome, outcome_present=outcome is not None, details_present=text_len(details) > 0,
        outcome_reasoning_present=text_len(g("outcome_reasoning")) > 0,
        statutes_n=len(stat), statutes_named_n=len(named), statutes_bare_n=len(bare), statutes_bare_dup_n=len(bare_dup),
        citations_n=len(cites), citations_named_n=len(cite_named), citations_bare_reporter_n=len(cite_bare_rep),
        concepts_n=len(concepts), concepts="|".join(concepts),
        pet_args_n=len(pet), res_args_n=len(res),
        petitioner=pp[:120], respondent=rn[:120], parties_present=bool(pp and rn),
        role_swap_candidate=swap_cand, outcome_inversion_candidate=inv,
        reasoning_len=(g("reasoning_len") if isinstance(g("reasoning_len"), (int, float)) else np.nan),
    ))
df = pd.DataFrame(rows)
print(df.shape); df.head(3)

## Cell 5 — Integrity + coverage

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z*z/n; c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return (100*(c-h), 100*(c+h))

S = {}   # everything that goes into summary_metrics.json
S["integrity"] = dict(
    json_files=len(files), loaded=len(records), load_errors=len(load_errors),
    unique_case_ids=int(df.case_id.nunique()), duplicate_case_ids=int(df.case_id.duplicated().sum()),
    missing_case_id=int((df.case_id == "").sum()), cases_before_2000=int((df.year < 2000).sum()),
    cases_year_unparsed=int(df.year.isna().sum()),
)
S["method_counts"] = df.method.value_counts().to_dict()
S["fallback"] = dict(
    regex_fallback_cases=int(df.is_fallback.sum()), pct=round(100*df.is_fallback.mean(), 2),
    call1_ok_pct=round(100*df.call1_ok.mean(), 2), call2_ok_pct=round(100*df.call2_ok.mean(), 2),
)
print(json.dumps(S["integrity"], indent=1)); print(json.dumps(S["fallback"], indent=1))
print("\nCases per year:"); yc = df.year.value_counts().sort_index(); print(yc.to_string())
S["cases_per_year"] = {int(k): int(v) for k, v in yc.items()}
print("\nCases per band:"); print(df.band.value_counts().to_string())
S["cases_per_band"] = df.band.value_counts().to_dict()

## Cell 6 — Field completeness (populated %, 95% CI) overall, by band, by year

In [ ]:
COMPLETE = {
    "title": "title_present", "issue": "issue_present", "facts": "facts_present", "case summary": "summary_present",
    "reasoning summary": "reasoning_summary_present", "principles": "principles_present", "outcome": "outcome_present",
    "decision details": "details_present", "outcome reasoning": "outcome_reasoning_present",
    "statutes": df.statutes_n > 0, "citations": df.citations_n > 0, "legal concepts": df.concepts_n > 0,
    "petitioner args": df.pet_args_n > 0, "respondent args": df.res_args_n > 0, "parties (both sides)": "parties_present",
}
def col(v): return df[v] if isinstance(v, str) else v

rows = []
for name, v in COMPLETE.items():
    s = col(v); k, n = int(s.sum()), len(s); lo, hi = wilson(k, n)
    rows.append(dict(field=name, populated=k, n=n, populated_pct=round(100*k/n, 1), ci_lo=round(lo, 1), ci_hi=round(hi, 1),
                     **{b: round(100*col(v)[df.band == b].mean(), 1) for b, _, _ in YEAR_BANDS}))
comp = pd.DataFrame(rows); print(comp.to_string(index=False))
S["completeness"] = comp.to_dict("records")

by_year = pd.DataFrame({name: col(v).groupby(df.year).mean().mul(100).round(1) for name, v in COMPLETE.items()})
print("\nPopulated % by year:"); print(by_year.to_string())
S["completeness_by_year"] = {int(y): r for y, r in by_year.to_dict("index").items()}

## Cell 7 — Outcome metrics + inversion *candidates*

In [ ]:
oc = df.outcome.fillna("MISSING").value_counts()
print("Outcome distribution:"); print((oc / len(df) * 100).round(2).astype(str).add("%").to_string())
S["outcome_distribution"] = oc.to_dict()
ob = pd.crosstab(df.band, df.outcome.fillna("MISSING"), normalize="index").mul(100).round(1)
print("\nOutcome % by band:"); print(ob.to_string()); S["outcome_by_band"] = ob.to_dict("index")
oy = pd.crosstab(df.year, df.outcome.fillna("MISSING"), normalize="index").mul(100).round(1)
S["outcome_by_year"] = {int(y): r for y, r in oy.to_dict("index").items()}

inv = df[df.outcome_inversion_candidate.notna()]
k = len(inv); lo, hi = wilson(k, len(df))
print(f"\nInversion CANDIDATES (keyword heuristic, NOT confirmed errors): {k} ({100*k/len(df):.2f}%, CI {lo:.2f}-{hi:.2f})")
print(inv.outcome_inversion_candidate.value_counts().to_string())
S["inversion_candidates"] = dict(n=k, pct=round(100*k/len(df), 2), ci=[round(lo, 2), round(hi, 2)],
                                 types=inv.outcome_inversion_candidate.value_counts().to_dict(),
                                 sample_case_ids=inv.case_id.head(50).tolist())

## Cell 8 — Statute metrics

In [ ]:
has = df[df.statutes_n > 0]
S["statutes"] = dict(
    cases_with_statutes=int(len(has)),
    entries_total=int(df.statutes_n.sum()), mean_per_case=round(df.statutes_n.mean(), 2),
    median_per_case=float(df.statutes_n.median()), p95_per_case=float(df.statutes_n.quantile(.95)), max_per_case=int(df.statutes_n.max()),
    named_entries=int(df.statutes_named_n.sum()), bare_section_entries=int(df.statutes_bare_n.sum()),
    bare_entries_pct=round(100*df.statutes_bare_n.sum()/max(df.statutes_n.sum(), 1), 1),
    bare_duplicates_of_named=int(df.statutes_bare_dup_n.sum()),
    cases_with_bare_duplicates=int((df.statutes_bare_dup_n > 0).sum()),
    cases_with_any_bare=int((df.statutes_bare_n > 0).sum()),
    cases_fully_named_pct=round(100*((has.statutes_bare_n == 0).mean()), 1),
)
print(json.dumps(S["statutes"], indent=1))
print("\nTop 5 cases by statute count:"); print(df.nlargest(5, "statutes_n")[["case_id", "statutes_n"]].to_string(index=False))
# most common statute strings (for sanity-checking hallucination patterns)
sc = Counter(item_text(x) for r in records.values() for x in as_list(get(r, "statutes")[0]))
S["top_statutes"] = sc.most_common(40); print("\nTop 15 statute strings:", sc.most_common(15))

## Cell 9 — Citation metrics

In [ ]:
S["citations"] = dict(
    cases_with_citations=int((df.citations_n > 0).sum()), entries_total=int(df.citations_n.sum()),
    mean_per_case=round(df.citations_n.mean(), 2), median_per_case=float(df.citations_n.median()),
    named_entries=int(df.citations_named_n.sum()), bare_reporter_entries=int(df.citations_bare_reporter_n.sum()),
    named_share_pct=round(100*df.citations_named_n.sum()/max(df.citations_n.sum(), 1), 1),
    cases_with_zero_named=int(((df.citations_n > 0) & (df.citations_named_n == 0)).sum()),
)
print(json.dumps(S["citations"], indent=1))
# empty citations vs judgment length (is empty plausible?)
if df.reasoning_len.notna().any():
    q = df.assign(lenbin=pd.qcut(df.reasoning_len, 4, duplicates="drop")).groupby("lenbin", observed=True).apply(lambda g: pd.Series(
        dict(n=len(g), citations_empty_pct=round(100*(g.citations_n == 0).mean(), 1), concepts_empty_pct=round(100*(g.concepts_n == 0).mean(), 1),
             principles_empty_pct=round(100*(~g.principles_present).mean(), 1), reasoning_summary_null_pct=round(100*(~g.reasoning_summary_present).mean(), 1))))
    print("\nEmpty rates by reasoning-text length quartile:"); print(q.to_string())
    S["empty_by_reasoning_length_quartile"] = {str(k): v for k, v in q.to_dict("index").items()}
else:
    print("No reasoning-length field found, skipping length-quartile analysis")

## Cell 10 — Legal concepts

In [ ]:
cc = Counter(c for s in df.concepts for c in s.split("|") if c)
S["concepts"] = dict(
    cases_with_concepts=int((df.concepts_n > 0).sum()), mean_per_case=round(df.concepts_n.mean(), 2),
    distinct_concepts=len(cc),
    top1_concept_share_of_cases_pct=round(100*max(cc.values())/len(df), 1) if cc else None,
    frequency=cc.most_common(),
    rare_concepts_lt_20_cases=[c for c, n in cc.items() if n < 20],
    basic_structure_cases=cc.get("basic_structure", 0), right_to_privacy_cases=cc.get("right_to_privacy", 0),
)
print(json.dumps({k: v for k, v in S["concepts"].items() if k != "frequency"}, indent=1))
print("\nFull frequency table:"); [print(f"  {c:40s} {n:6d}  ({100*n/len(df):.1f}%)") for c, n in cc.most_common()]

## Cell 11 — Arguments + parties (role-swap *candidates*)

In [ ]:
sw = df[df.role_swap_candidate == True]; testable = df.role_swap_candidate.notna()
lo, hi = wilson(len(sw), int(testable.sum()))
S["arguments"] = dict(
    pet_args_mean=round(df.pet_args_n.mean(), 2), res_args_mean=round(df.res_args_n.mean(), 2),
    pet_empty_pct=round(100*(df.pet_args_n == 0).mean(), 1), res_empty_pct=round(100*(df.res_args_n == 0).mean(), 1),
    both_empty_pct=round(100*((df.pet_args_n == 0) & (df.res_args_n == 0)).mean(), 1),
    one_sided_pct=round(100*((df.pet_args_n == 0) ^ (df.res_args_n == 0)).mean(), 1),
)
S["parties"] = dict(
    both_sides_present_pct=round(100*df.parties_present.mean(), 1),
    role_swap_candidates=int(len(sw)), testable_cases=int(testable.sum()),
    role_swap_candidate_pct=round(100*len(sw)/max(int(testable.sum()), 1), 2), ci=[round(lo, 2), round(hi, 2)],
    sample_case_ids=sw.case_id.head(50).tolist(),
)
print(json.dumps(S["arguments"], indent=1)); print(json.dumps({k: v for k, v in S["parties"].items() if k != "sample_case_ids"}, indent=1))
print("\nNOTE: role-swap is a caption-order heuristic - a candidate is not a confirmed error.")

## Cell 12 — Save outputs

In [ ]:
df.to_csv(os.path.join(REPORT_DIR, "chunk_stats.csv"), index=False)
with open(os.path.join(REPORT_DIR, "summary_metrics.json"), "w") as fh:
    json.dump(S, fh, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
if load_errors:
    pd.DataFrame(load_errors, columns=["file", "error"]).to_csv(os.path.join(REPORT_DIR, "load_errors.csv"), index=False)
print("Saved to", REPORT_DIR, ":", os.listdir(REPORT_DIR))
print("\nUpload chunk_stats.csv and summary_metrics.json to Claude for the full report.")